# EDA — BIS central bank policy rates (`WS_CBPOL`, CSV flat)

Exploratory analysis of the BIS bulk file before building the pipeline. The goal is to replace
assumptions with observed facts about structure, coding conventions, missing values, duplicates,
metadata, calendar density and recency, because those facts drive the `transform` and `snapshot`
design.

**Input:** `resources/WS_CBPOL_csv_flat.csv`, the unzipped *Central bank policy rates (CSV, flat)*
file from <https://data.bis.org/bulkdownload>. The `resources/` folder is gitignored.

**Run:** see `diagnostics/README.md`. Loading the whole file takes about 10 s and needs well under 1 GB of
RAM, because repeated text columns are stored as categories.

## 1. Setup

In [ ]:
from __future__ import annotations

import re
import time
from decimal import Decimal
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)

# The working directory is the notebook's folder; walk up to the repository root.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
DATA = ROOT / "resources" / "WS_CBPOL_csv_flat.csv"

# Areas used in the brief's example. "EA" is kept on purpose, to show it is not a BIS code.
BRIEF_AREAS = ["US", "EA", "GB", "JP", "CH"]
ALIASES = {"EA": "XM"}  # finding of §7: the euro area is coded XM
BRIEF_CODES = [ALIASES.get(a, a) for a in BRIEF_AREAS]
REPORT_START = "2015-01-01"

print(f"{DATA}\n{DATA.stat().st_size / 1e6:,.1f} MB")

## 2. Raw header and first rows

Before typing anything, look at the file as it is.

In [ ]:
raw_header = pd.read_csv(DATA, nrows=0).columns.tolist()
# Headers are either "CODE" or "CODE:Label". str.partition handles both forms.
HEADER = pd.DataFrame(
    [(h, *h.partition(":")[::2]) for h in raw_header], columns=["raw", "code", "label"]
)
display(HEADER)

In [ ]:
pd.read_csv(DATA, nrows=3, dtype=str, keep_default_na=False).T

## 3. Load the full file

- All columns are read as **strings**, with `na_filter=False`, so the raw encodings (for example
  the literal `"NaN"`) stay visible.
- Repeated attribute columns use the `category` dtype to keep memory low.
- Columns are renamed to their bare codes.

In [ ]:
CODES = HEADER["code"].tolist()
STRING_COLS = {"TIME_PERIOD", "OBS_VALUE"}

t0 = time.perf_counter()
raw = pd.read_csv(
    DATA,
    header=0,
    names=CODES,
    dtype={c: ("string" if c in STRING_COLS else "category") for c in CODES},
    na_filter=False,
)
print(f"loaded {len(raw):,} rows x {raw.shape[1]} cols in {time.perf_counter() - t0:.1f}s, "
      f"{raw.memory_usage(deep=True).sum() / 1e6:,.0f} MB in memory")

## 4. Column profile

For each column: the number of distinct values, the share of empty strings, and whether values
follow the SDMX `code: label` convention.

In [ ]:
CODE_LABEL_RE = re.compile(r"^[^:\s]+: ")


def profile(col: pd.Series) -> dict[str, object]:
    values = pd.Series(col.astype("string").unique())
    non_empty = values[values != ""]
    coded = non_empty.str.match(CODE_LABEL_RE).mean() if len(non_empty) else np.nan
    return {
        "nunique": col.nunique(),
        "empty_share": (col.astype("string") == "").mean(),
        "coded_share_of_distinct": coded,
        "example": non_empty.iloc[0][:70] if len(non_empty) else "",
    }


col_profile = pd.DataFrame({c: profile(raw[c]) for c in raw.columns}).T
display(col_profile)

Columns where every row holds the same value carry no information per row. Even so, the
pipeline should **assert** their expected values, so that a change in the source is noticed.

In [ ]:
constant = {c: raw[c].iloc[0] for c in raw.columns if raw[c].nunique() == 1}
pd.Series(constant, name="single value")

## 5. Decode `code: label` columns into a working frame

In [ ]:
def split_coded(col: pd.Series) -> tuple[pd.Series, pd.Series]:
    """Split a categorical 'code: label' column into code and label (done per category)."""
    cats = col.cat.categories.to_series()
    parts = cats.str.partition(": ")
    code_map, label_map = parts[0], parts[2]
    return col.map(code_map).astype("category"), col.map(label_map).astype("category")


df = pd.DataFrame(index=raw.index)
df["freq"], _ = split_coded(raw["FREQ"])
df["area"], df["area_name"] = split_coded(raw["REF_AREA"])
df["status"], _ = split_coded(raw["OBS_STATUS"])
df["period"] = raw["TIME_PERIOD"]
df["raw_value"] = raw["OBS_VALUE"]
df["value"] = pd.to_numeric(raw["OBS_VALUE"], errors="coerce")
df["title"] = raw["TITLE"]
df.head()

## 6. Frequencies and time-period formats

In [ ]:
display(df["freq"].value_counts().rename("rows").to_frame())

period_shape = df["period"].str.replace(r"\d", "9", regex=True)
display(pd.crosstab(df["freq"], period_shape))

In [ ]:
# Parse dates using the format implied by each frequency. Any NaT means an unparseable period.
df["date"] = pd.NaT
is_d, is_m = df["freq"].eq("D"), df["freq"].eq("M")
df.loc[is_d, "date"] = pd.to_datetime(df.loc[is_d, "period"], format="%Y-%m-%d", errors="coerce")
df.loc[is_m, "date"] = pd.to_datetime(df.loc[is_m, "period"], format="%Y-%m", errors="coerce")
df["date"] = pd.to_datetime(df["date"])
print("unparseable periods:", df["date"].isna().sum())

## 7. Series inventory

A *series* is a `(freq, area)` pair.

- Which areas have daily data?
- Which areas have monthly data only?
- Is the euro area coded `EA` or `XM`?

In [ ]:
inventory = (
    df.groupby(["freq", "area"], observed=True)
    .agg(
        area_name=("area_name", "first"),
        first=("date", "min"),
        last=("date", "max"),
        rows=("date", "size"),
    )
    .reset_index()
)
by_area = inventory.pivot(index="area", columns="freq", values="rows").fillna(0).astype(int)
print(f"areas: {by_area.shape[0]}  with daily: {(by_area['D'] > 0).sum()}  "
      f"monthly only: {((by_area['D'] == 0) & (by_area['M'] > 0)).sum()}")
display(by_area[by_area["D"] == 0])

for code_ in BRIEF_AREAS:
    hit = inventory[inventory["area"].eq(code_)]
    print(f"{code_}: {'present' if len(hit) else 'NOT PRESENT'}")

display(inventory[inventory["area_name"].str.contains("euro", case=False)])

## 8. Observation values and missing values

- How are missing values encoded?
- Does `OBS_STATUS` agree with the value?
- Are any values non-numeric?

In [ ]:
non_numeric = df.loc[df["value"].isna() & ~df["raw_value"].isin(["NaN", ""]), "raw_value"]
print("non-numeric raw values (excluding 'NaN'/empty):", non_numeric.value_counts().head(10).to_dict())

raw_kind = np.select(
    [df["raw_value"].eq(""), df["raw_value"].eq("NaN"), df["value"].notna()],
    ["empty", "'NaN' string", "number"],
    default="other",
)
display(pd.crosstab([df["freq"], df["status"]], raw_kind, margins=True))

Note that `float("NaN")` parses without error. A cast alone would therefore let missing values
through as valid floats. Validity has to require **both** a status other than `M` **and** a finite
value.

In [ ]:
df["is_valid"] = df["status"].ne("M") & np.isfinite(df["value"])
d = df[df["freq"].eq("D")].copy()
d["area"] = d["area"].astype(str)  # plain str for group/merge convenience
print(f"daily rows: {len(d):,}  valid: {d['is_valid'].sum():,}  ({d['is_valid'].mean():.1%})")

In [ ]:
v = d[d["is_valid"]]
value_stats = v.groupby("area")["value"].agg(["min", "max", "last"]).sort_values("max")
print("areas with negative rates:", value_stats.index[value_stats["min"] < 0].tolist())
display(value_stats.tail(8))

## 9. Duplicates

- **Exact duplicate:** same key and same value.
- **Conflicting duplicate:** same key, different values.

The key is `(freq, area, period)`.

In [ ]:
key = ["freq", "area", "period"]
dup_mask = df.duplicated(key, keep=False)
dups = df[dup_mask]
exact = dups.duplicated([*key, "raw_value"], keep=False).sum()
print(f"rows sharing a key: {dup_mask.sum():,}  exact duplicates: {exact:,}  "
      f"conflicting: {dup_mask.sum() - exact:,}")
display(dups.head())

If both counts are zero, the current file is clean. Duplicate handling is still a requirement of
the brief, and it will be tested on synthetic fixtures.

## 10. Series attributes and metadata

Is each attribute constant within a series? If so, it can be stored once per series in a separate
metadata table, rather than repeated on every observation.

In [ ]:
ATTRS = ["TITLE", "UNIT_MEASURE", "UNIT_MULT", "DECIMALS", "COMPILATION", "SOURCE_REF",
         "SUPP_INFO_BREAKS", "TIME_FORMAT", "OBS_CONF", "OBS_PRE_BREAK"]
series_id = raw["FREQ"].astype(str) + "|" + raw["REF_AREA"].astype(str)
varying = raw[ATTRS].groupby(series_id, observed=True).nunique().gt(1).sum()
display(varying.rename("series where the attribute varies").to_frame())

In [ ]:
titles = raw["TITLE"].cat.categories.to_series()
print("titles with leading/trailing whitespace:", (titles != titles.str.strip()).sum(), "of", len(titles))
print("example:", repr(titles.iloc[0]))
print("UNIT_MEASURE:", raw["UNIT_MEASURE"].unique().tolist())
print("UNIT_MULT:", raw["UNIT_MULT"].unique().tolist(), " DECIMALS:", raw["DECIMALS"].unique().tolist())

In [ ]:
# SUPP_INFO_BREAKS: free text describing regime changes and discontinuations. It is useful context
# for the report's limitations section.
breaks = raw.loc[raw["FREQ"].str.startswith("D"), ["REF_AREA", "SUPP_INFO_BREAKS"]].drop_duplicates()
breaks["len"] = breaks["SUPP_INFO_BREAKS"].astype(str).str.len()
display(breaks.sort_values("len", ascending=False).head(5))

## 11. Calendar density: calendar days or business days?

This decides how "12 months earlier" must be looked up. If a series has no row on weekends, an
exact-date lookup will often fail.

In [ ]:
recent = d[d["date"] >= "2020-01-01"].copy()
recent["weekend"] = recent["date"].dt.day_of_week >= 5
density = recent.groupby("area").agg(
    rows_per_year=("date", lambda s: len(s) / (s.max() - s.min()).days * 365.25),
    weekend_share=("weekend", "mean"),
    valid_share=("is_valid", "mean"),
)
density["calendar"] = np.where(density["weekend_share"] > 0.2, "calendar days", "business days")
display(density["calendar"].value_counts())
display(density.loc[BRIEF_CODES])

In [ ]:
# Where do status-M rows fall? On weekends, holidays, or in blocks?
m_by_dow = pd.crosstab(recent["date"].dt.day_name(), recent["status"], normalize="index")
display(m_by_dow.reindex(["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]))

In [ ]:
# Longest gap between consecutive valid observations since 2000, per series.
gv = d[d["is_valid"] & (d["date"] >= "2000-01-01")].sort_values(["area", "date"])
gv["gap_days"] = gv.groupby("area")["date"].diff().dt.days
longest = gv.loc[gv.groupby("area")["gap_days"].idxmax().dropna(), ["area", "date", "gap_days"]]
display(longest.sort_values("gap_days", ascending=False).head(10))

## 12. Recency: series end on different dates

The brief warns not to assume a shared latest date. This section measures how far apart the
latest dates are, and whether the last row of a series can be invalid.

In [ ]:
last_row = d.groupby("area")["date"].max().rename("last_row")
last_valid = d[d["is_valid"]].groupby("area")["date"].max().rename("last_valid")
recency = pd.concat([last_row, last_valid], axis=1)
dataset_max = recency["last_valid"].max()
recency["lag_days"] = (dataset_max - recency["last_valid"]).dt.days
recency["trailing_invalid_rows"] = [
    int(((d["area"] == a) & (d["date"] > lv)).sum()) for a, lv in recency["last_valid"].items()
]
print("dataset max valid date:", dataset_max.date())
display(recency["last_valid"].dt.date.value_counts().head(8).rename("areas").to_frame())
display(recency.sort_values("lag_days", ascending=False).head(15))

## 13. Prototype of the 12-month look-back

Steps, per area:

1. Take the latest valid observation.
2. Compute the target date: 12 calendar months earlier.
3. Take the last valid observation **on or before** the target date (`merge_asof`, backward).
4. Compute the change in basis points.

The table also checks whether the target date itself has a row, and compares float arithmetic
with decimal arithmetic for the change.

In [ ]:
latest = (
    v.sort_values("date").groupby("area").tail(1)
    .set_index("area")[["area_name", "date", "value"]]
    .rename(columns={"date": "latest_date", "value": "latest"})
)
latest["target_date"] = [ts - pd.DateOffset(months=12) for ts in latest["latest_date"]]

prior = pd.merge_asof(
    latest.reset_index().sort_values("target_date"),
    v[["area", "date", "value"]].sort_values("date").rename(columns={"date": "prior_date", "value": "prior"}),
    left_on="target_date", right_on="prior_date", by="area", direction="backward",
).set_index("area")

prior["exact_target_row"] = [
    ((v["area"] == a) & (v["date"] == t)).any() for a, t in prior["target_date"].items()
]
prior["prior_lag_days"] = (prior["target_date"] - prior["prior_date"]).dt.days
prior["change_bps_float"] = (prior["latest"] - prior["prior"]) * 100
prior["change_bps_decimal"] = [
    (Decimal(str(a)) - Decimal(str(b))) * 100 if pd.notna(b) else None
    for a, b in zip(prior["latest"], prior["prior"], strict=True)
]
float_noise = prior["change_bps_float"].notna() & (prior["change_bps_float"] != prior["change_bps_float"].round(6))
print("areas where float arithmetic shows noise:", prior.index[float_noise].tolist())
print("areas without an exact row on the target date:", prior.index[~prior["exact_target_row"]].tolist())

display(prior.loc[BRIEF_CODES])

## 14. Consistency between daily and monthly series

Each monthly series is labelled *End of period*. Does its value equal the last valid daily value of
that month? This makes a useful cross-check, and it shows how the two frequencies relate.

In [ ]:
monthly = df[df["freq"].eq("M") & df["is_valid"]][["area", "date", "value"]].copy()
monthly["area"] = monthly["area"].astype(str)
monthly["ym"] = monthly["date"].dt.to_period("M")

daily_eom = v.assign(ym=v["date"].dt.to_period("M")).sort_values("date").groupby(["area", "ym"]).tail(1)
cmp = monthly.merge(daily_eom[["area", "ym", "value"]], on=["area", "ym"], suffixes=("_m", "_d"))
cmp["diff_bps"] = (cmp["value_m"] - cmp["value_d"]).abs() * 100
agree = cmp.groupby("area")["diff_bps"].agg(months="size", mismatches=lambda s: (s > 0.01).sum())
print(f"overlapping area-months: {len(cmp):,}  mismatching: {(cmp['diff_bps'] > 0.01).sum():,}")
display(agree.sort_values("mismatches", ascending=False).head(10))

## 15. Rate moves

Policy rates are step functions. Counting the moves and their sizes gives a quick sanity check,
and the largest moves are candidates for implausible-value checks.

In [ ]:
moves = v[v["date"] >= "2000-01-01"].sort_values(["area", "date"]).copy()
moves["move_bps"] = moves.groupby("area")["value"].diff() * 100
moves = moves[moves["move_bps"].fillna(0) != 0]
print(f"rate moves since 2000: {len(moves):,}")
display(moves["move_bps"].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).to_frame())
display(moves.reindex(moves["move_bps"].abs().sort_values(ascending=False).index)
        .head(10)[["area", "date", "value", "move_bps"]])

## 16. Daily policy rates of the brief's example areas

In [ ]:
plot_areas = BRIEF_CODES
fig, ax = plt.subplots(figsize=(11, 5))
for area in plot_areas:
    s = v[(v["area"] == area) & (v["date"] >= REPORT_START)].set_index("date")["value"]
    ax.plot(s.index, s.values, drawstyle="steps-post", label=f"{area} ({s.index.max():%Y-%m-%d})")
ax.axhline(0, color="grey", linewidth=0.8)
ax.set(title="Central bank policy rates, daily (BIS WS_CBPOL)", ylabel="Per cent per year")
ax.legend(title="area (latest valid date)")
ax.grid(alpha=0.3)
plt.show()

## 17. Findings → design implications

Recorded from a run on 2026-11-17 against the file published on 2026-11-11. Re-run the notebook
when the source changes.

| # | Finding (section) | Design implication |
|---|---|---|
| 1 | Headers are `CODE:Label`, except `STRUCTURE`, `STRUCTURE_ID` and `ACTION`, which have no label. Coded values are `code: label` (§2, §4). | One helper splits both forms; keep both the code and the label. |
| 2 | 734,939 rows: 709,767 daily (`D`) and 25,172 monthly (`M`). All 49 areas have a daily series. Periods are `YYYY-MM-DD` (daily) and `YYYY-MM` (monthly), and all parse (§6, §7). | Filter on the code `D`. Parse dates with the format of each frequency, and reject rows that do not match. |
| 3 | The euro area is **`XM`**. `EA`, used in the brief's example, is not in the file (§7). | An alias table `EA → XM`. Unknown codes return a status and do not stop the other areas. |
| 4 | Missing values are the string `"NaN"` with status `M`: 70,026 daily rows and 53 monthly rows. There are no empty values and no other non-numeric strings. The status and the value always agree (§8). | Valid means status ≠ `M` **and** a finite value. `float("NaN")` parses, so a cast alone would accept these rows. |
| 5 | No duplicate `(freq, area, period)` keys (§9). | Keep the duplicate rules anyway and test them on synthetic fixtures. |
| 6 | Unit (per cent per year), multiplier (×1) and decimals (4) are the same across the whole file. No attribute varies within a series. All 98 titles have a leading space. `SUPP_INFO_BREAKS` holds up to about 1,100 characters of series history (§4, §10). | A per-series metadata table. Strip titles. Assert that attributes are constant within a series. Use the break notes in the report's limitations section. |
| 7 | Since 2020, 29 daily series have rows for every calendar day and 11 for business days only. US is calendar; GB, JP and CH are business-day. About 60% of weekend rows are status `M` (§11). | "12 months earlier" must use the last valid value on or before the target date. 8 areas, GB among them, have no row on their target date (§13). |
| 8 | Long gaps between valid values: JP 1,817 days (2001–2006), HR 689 days (§11). | Flag a prior value far older than its target date (`stale_prior`). Do not hide it. |
| 9 | Latest valid dates differ. 18 areas end on 2026-11-10, others days or weeks earlier. Some are far behind: MA 71 days, AR 488 days, and 9 legacy euro members end in 1998–2000. HR's last 166 rows are invalid (§12). | Each area is evaluated on its own latest **valid** date. Flag stale series (`stale_latest`). |
| 10 | Float arithmetic adds noise to the bps change for 6 areas (§13). | Compute the change with `Decimal`. |
| 11 | Historic extremes: BR up to 790,799%, HR 2,790%, AR 557%. Since 2000, the largest single moves are about ±53,000 bp (AR, 2001–02) (§8, §15). | Range checks should **flag**, not drop. Charts should start at a sensible date (the report defaults to 2015). |
| 12 | Monthly end-of-period values equal the last daily value of the month in 25,107 of 25,119 cases. The remaining 12 differ (§14). | An optional data-quality cross-check. Note it as a known source limitation. |

Snapshot for the brief's example areas on this file (§13): US 3.875 (unchanged), GB 4.00 (unchanged,
prior taken from 2025-11-07), JP 1.25 (+75 bp), CH 0.00 (unchanged), euro area 2.50 (+50 bp).